# 📌 Projet P12 : Détection de Faux Billets par Machine Learning

---
## **0. Contexte et Veille** *(Bloc BC04)*
- **Contexte** : Description du projet ONFM et des enjeux.
- **Veille** :
  - Standards **EMV** pour la détection de fraude.
  - Benchmark des outils de classification (Scikit-learn, PyTorch, etc.).
- **Objectifs** : Liste des objectifs techniques et métiers.
  -  mettre à disposition des équipes de l'ONFM une application de machine learning.
  -  faire une prédiction sur la nature des billets (vrai billet ou faux billet) afin d'accélérer le processus de détection de faux billets et d'améliorer la sécurité des transactions financières.L’objectif métier est surtout de limiter les faux billets classés comme vrais.

---

# 1) Import des bibliothèques nécessaires

In [ ]:
# Utilitaires
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Données et visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Partitions
from sklearn.model_selection import train_test_split

# Machine Learning - Preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler # Standardisation des données numériques
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder # Encodage des données catégorielles
from sklearn.impute import SimpleImputer, KNNImputer # Imputation des valeurs manquantes

# Machine Learning - Pipeline
from sklearn.pipeline import Pipeline

# Modèles de classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Machine Learning - Evaluation  des modèles
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay

# Machine Learning - Optimisation des hyperparamètres
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

print("✅ Librairies chargées ")

# 2) Chargement et Structuration des Données (Bloc BC01 et BC02)

Le jeu de données fourni est un fichier CSV contenant des informations sur des billets de banque (billets.csv). Chaque ligne du fichier représente un billet, et chaque colonne représente une caractéristique du billet. La variable cible est "is_genuine", qui indique si le billet est authentique (True) ou non authentique (False).



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "../data/raw/billets.csv"

if not os.path.exists(DATA_PATH):
    print(f"❌ Attention : le fichier '{DATA_PATH}' n'existe pas.")

try:
    df = pd.read_csv(DATA_PATH, sep=";") # Chargement du dataset ONFM
    print("✅ Données chargées, voici les dimensions du dataset:", df.shape)
    display(df.head()) # Structuration des données (DataFrame Pandas)
except Exception as e:
    print("❌ Erreur lors du chargement :", e)
  


# 3) Analyse exploratoire (EDA)


Objectif EDA : comprendre la qualité des données avant modélisation (dimensions, types, statistiques, valeurs manquantes, doublons, équilibre de la cible).

## 3.0 - Premier aperçu des données 

In [ ]:
#dimensions
print (f"-" * 60 + " PREMIER APERCU DES DONNEES " + "-" * 60)
print (f"Les dimensions du dataframe sont: \n{df.shape}"+ "\n" )

#types
print (f"Les types des colonnes sont: \n{df.dtypes}"+ "\n")

#statistiques descriptives
print (f"Les statistiques descriptives sont:\n:{df.describe()}"+ "\n")

#valeurs manquantes
print (f"Les valeurs manquantes sont:\n{df.isnull().sum()}"+ "\n")

#doublons
print (f"Les doublons sont:{df.duplicated().sum()}"+ "\n")

#répartition de la Target
print (f"La répartition de la Target (is_genuine (est authentique)) :\n{df['is_genuine'].value_counts()}"+ "\n")

print ("-" * 120)


Conclusion : la variable `margin_low` contient des valeurs manquantes. Elles doivent être imputées avant l'entraînement, car les modèles scikit-learn utilisés ici ne peuvent pas apprendre directement avec des `NaN`. L'imputation par la médiane est cohérente avec des mesures numériques et limite l'influence des valeurs extrêmes.

## 3.1 - Corrélations entre variables
Cette étape permet d'identifier les variables les plus informatives pour la prédiction de l'authenticité.

In [ ]:
# Calculer la matrice de corrélation sur les variables numériques
pcorr = df.select_dtypes(include='number').corr()

# Afficher la matrice de corrélation sous forme de heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(pcorr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matrice de corrélation (variables numériques)')
plt.show()

Dans notre cas, nous voulons prédire si un billet est authentique (is_genuine=True) ou non authentique (is_genuine=False) à partir des variables explicatives que nous allons sélectionner.

 Pour cela, nous décidons aussi d'ajouter une variable "faux_billet" à notre dataframe qui sera égale à 1 si le billet n'est pas authentique et 0 sinon. 

Nous allons ensuite utiliser cette variable binaire comme variable cible pour entraîner notre modèle de classification binaire pour améliorer les performances de nos modèles à tester.

In [ ]:
# Créer une cible binaire temporaire si elle n'existe pas encore (1 = faux, 0 = vrai)
if 'faux_billet' not in df.columns:
    if pd.api.types.is_bool_dtype(df['is_genuine']):
        df['faux_billet'] = (~df['is_genuine']).astype(int)
    else:
        df['faux_billet'] = (
            df['is_genuine']
            .astype(str)
            .str.strip() # strip permet de supprimer les espaces avant et après la chaîne de caractères
            .str.lower() # convention de mise en minuscules pour éviter les erreurs de casse
            .map({'false': 1, 'true': 0})
            .astype(int)
        )

# Visualiser la distribution de la cible par rapport à la longueur
fig = plt.figure(figsize=(6, 6))
plt.scatter(df['length'], df['faux_billet'], alpha=0.5)
plt.grid()
plt.xlabel('Longueur du billet (mm)')
plt.ylabel('Authenticité (0 = authentique, 1 = faux)')
plt.title('Distribution de faux billets selon la longueur')
plt.show()

Conclusion : certaines variables sont corrélées, ce qui invite à surveiller la redondance d'information. Ce n'est pas bloquant pour tous les modèles : la régression logistique peut y être sensible, tandis que Random Forest l'est moins. L'ACP est donc testée comme outil exploratoire, mais la sélection finale se fera sur les performances validées.

## 3.2 - Recherche de la présence de Outliers

On utilise la méthode IQR Globale pour détecter les outliers sur toutes les variables numériques du dataframe. On constate que certaines variables présentent des valeurs aberrantes qui pourraient influencer négativement la performance de notre modèle de classification binaire.

In [ ]:
# Approche descriptive globale IQR sur les variables explicatives numériques
features_num = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
num = df[features_num].copy()

q1 = num.quantile(0.25)
q3 = num.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# True/False par cellule, puis agrégation au niveau ligne
mask_iqr = num.lt(lower, axis=1) | num.gt(upper, axis=1)
df['nb_outliers_iqr'] = mask_iqr.sum(axis=1)
df['pct_outliers_iqr'] = df['nb_outliers_iqr'] / len(features_num)

# Règle globale (à ajuster): ligne considérée outlier si >= 2 variables aberrantes
df['outlier_global_iqr'] = df['nb_outliers_iqr'] >= 2

print('Nombre de lignes outliers (IQR global):', int(df['outlier_global_iqr'].sum()))
display(df.loc[df['outlier_global_iqr'], features_num + ['nb_outliers_iqr', 'pct_outliers_iqr']].head())

Conclusion : la méthode IQR signale quelques observations atypiques. Leur nombre reste faible ; elles ne sont donc pas supprimées automatiquement. Le pipeline traitera surtout les valeurs manquantes et l'échelle des variables, puis les modèles seront évalués sur leur capacité à généraliser.

## 3.3 - ACP pour visualiser la structure des données avant Kmeans (algorithme non supervisé)
On réduit la dimension pour projeter les billets en 2D et observer la séparation potentielle entre classes.

In [ ]:
# Après imputation et standardisation des variables explicatives, on peut faire une ACP
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

FEATURES = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
TARGET = 'faux_billet'

# Créer la cible si nécessaire
if TARGET not in df.columns:
    if pd.api.types.is_bool_dtype(df['is_genuine']):
        df[TARGET] = (~df['is_genuine']).astype(int)
    else:
        df[TARGET] = (
            df['is_genuine']
            .astype(str)
            .str.strip()
            .str.lower()
            .map({'false': 1, 'true': 0})
            .astype(int)
        )

# 1. Séparer features (X) et target (y)
X = df[FEATURES].copy()
y = df[TARGET].astype(int)

# 2. Imputer les valeurs manquantes avant standardisation
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# 3. Standardiser les variables explicatives avant l'ACP
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# 4. Appliquer l'ACP
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# 5. Analyser la variance expliquée
print('Variance expliquée par composante:', pca.explained_variance_ratio_)
print('Variance cumulée:', pca.explained_variance_ratio_.cumsum())

In [ ]:
# ACP pas très probante (pratiquement le nombre de variables prédictives), mais on peut réduire à 6 dimensions principales pour garder 95% de la variance expliquée et visualiser la forme des clusters
# Réduire à 95% de la variance expliquée
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

Conclusion : l'ACP montre qu'il faut conserver plusieurs composantes pour expliquer l'essentiel de la variance. La projection en 2D donne une lecture visuelle utile, mais elle ne suffit pas à elle seule pour décider du modèle final.

In [ ]:
# on visualise la separation des classes dans l'espace des 2 premieres composantes principales pour voir le comportement
import matplotlib.pyplot as plt

plt.scatter(X_pca[y == 1, 0], X_pca[y == 1, 1], label='faux billet', alpha=0.7)
plt.scatter(X_pca[y == 0, 0], X_pca[y == 0, 1], label='vrai billet', alpha=0.7)
plt.xlabel('Composante Principale 1')
plt.ylabel('Composante Principale 2')
plt.legend()
plt.title('Projection ACP des billets (2D)')
plt.show()

In [ ]:
#comparaison avec les variables scalées
plt.scatter(X_scaled[y == 1, 0], X_scaled[y == 1, 1], label='Classe 1')
plt.scatter(X_scaled[y == 0, 0], X_scaled[y == 0, 1], label='Classe 0')
plt.xlabel('Variable 1')
plt.ylabel('Variable 2')
plt.legend()
plt.show()

Conclusion ACP : la projection aide à visualiser la structure générale des données, mais la séparation entre vrais et faux billets n'est pas parfaitement lisible en deux dimensions. L'ACP reste donc exploratoire ; la décision finale doit reposer sur les métriques de classification.

# 4) Prétraitement et préparation des données

Objectif : transformer les données brutes en entrées exploitables par les modèles, sans fuite de données. La condition est de ne pas utiliser la variable cible pour le prétraitement des données et de ne pas utiliser les variables fortement corrélées entre elles pour éviter la multicolinéarité.

Pré-requis : les variables catégorielles doivent être encodées et les variables numériques doivent être normalisées pour que les modèles de classification binaire puissent les utiliser efficacement. ici nous avons déjà encodé les variables catégorielles et normalisé les variables numériques dans l'étape précédente.

D'abord, il faut séparer les variables explicatives et la cible :

X = les mesures du billet (longueur, hauteur, diagonales…)

y = l’étiquette (vrai ou faux)

➡️ Pourquoi ?
Parce que le modèle doit apprendre à prédire y en fonction de X.

### 4.1 Préparation des données

### 4.1.1 Séparation variables explicatives / cible

#### On normalise les variables numériques en excluant la variable cible  "is_genuine".
les modèles sont sensibles aux outliers, il est donc important de les détecter et de les traiter avant d'entraîner notre modèle de classification binaire.

On utilise le RobustScaler pour normaliser les données en utilisant la médiane et l'écart interquartile, ce qui est plus robuste aux valeurs aberrantes que la standardisation classique.

In [ ]:
# Creation de la cible puis normalisation des seules variables explicatives
from sklearn.preprocessing import RobustScaler

FEATURES = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
TARGET = 'faux_billet'

if pd.api.types.is_bool_dtype(df['is_genuine']):
    df[TARGET] = (~df['is_genuine']).astype(int)
else:
    df[TARGET] = (
        df['is_genuine']
        .astype(str)
        .str.strip()
        .str.lower()
        .map({'false': 1, 'true': 0})
        .astype(int)
    )

scaler = RobustScaler()
df_normalized = df.copy()
df_normalized[FEATURES] = scaler.fit_transform(df[FEATURES])

print(df_normalized[FEATURES + [TARGET]].head())
df_normalized.to_csv('../data/processed/billets_cleaned.csv', index=False)

print('df_normalized sauvegardé dans billets_cleaned.csv')

In [ ]:
# Vérification de la distribution de la variable cible
FEATURES = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
TARGET = 'faux_billet'

if 'df_normalized' not in globals():
    df_normalized = df.copy()

print(df[['is_genuine', TARGET]].head())

counts = df_normalized[TARGET].value_counts()
faux = counts.get(1, 0)
vrai = counts.get(0, 0)

print(df['is_genuine'].unique())
print(df_normalized[TARGET].value_counts())

# Visualiser la distribution de la variable cible
fig = plt.figure(figsize=(6, 6))
plt.bar(['Faux', 'Vrai'], [faux, vrai], alpha=0.5)
plt.grid()
plt.xlabel('Authenticité (0 = authentique, 1 = faux)')
plt.ylabel('Nombre de billets')
plt.title('Distribution de la variable cible "faux_billet"', fontdict={'fontsize': 14, 'fontweight': 'bold'})
plt.show()

#### On traite les outliers par une approche mixte qui les détecte sur tout le dataframe 

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# 1) Variables explicatives numériques métier
FEATURES = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
X = df[FEATURES].copy()

# 2) Prétraitement global (imputation + standardisation)
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

X_prep = preprocessor.fit_transform(X)

# 3) Détection d'outliers sur l'ensemble des variables
iso = IsolationForest(
    n_estimators=300, # nombre d'arbres dans la forêt
    contamination=0.03,  # à ajuster selon le contexte (1% à 5%)
    random_state=42
)
labels = iso.fit_predict(X_prep)  # -1 = outlier, 1 = inlier
scores = iso.decision_function(X_prep)

# 4) Résultats
df_outliers = df.copy()
df_outliers['is_outlier'] = (labels == -1)
df_outliers['outlier_score'] = scores

print('Nb outliers détectés :', int(df_outliers['is_outlier'].sum()))
display(df_outliers.loc[df_outliers['is_outlier'], FEATURES + ['outlier_score']].head())

On constate grâce à la méthode d'isolationForest la présence de 45 outliers sur toutes les variables numériques du dataframe. On décide de les traiter par imputation de la médiane pour ne pas perdre d'observations et éviter de biaiser notre modèle de classification binaire.

#### On vérifie par Z-Score l'absence des outliers après normalisation 

In [ ]:
# Vérification descriptive globale par Z-score (seuil = 3)
from scipy.stats import zscore

features_num = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
num = df[features_num].copy()

z = pd.DataFrame(
    np.abs(zscore(num, nan_policy='omit')), # calcul du Z-score absolu, nan_policy='omit' pour ignorer les NaN
    index=num.index,
    columns=num.columns,
 )

mask_z = z > 3
df['nb_outliers_z'] = mask_z.sum(axis=1)
df['pct_outliers_z'] = df['nb_outliers_z'] / len(features_num)
df['outlier_global_z'] = df['nb_outliers_z'] >= 2

print('Nombre de lignes outliers (Z-score global):', int(df['outlier_global_z'].sum()))
display(df.loc[df['outlier_global_z'], features_num + ['nb_outliers_z', 'pct_outliers_z']].head())

Conclusion : après imputation et standardisation, les données sont dans un format exploitable par les modèles. La standardisation ne supprime pas les outliers, mais elle met les variables sur une échelle comparable. Les observations atypiques restent donc à interpréter avec prudence lors de l'évaluation.

#### Vérification K-means sur les variables normalisées et standardisées

Attendus :
- K-means est un algorithme non supervisé : il crée des groupes selon les distances, sans utiliser la variable cible.
- On peut comparer ses clusters aux vraies classes uniquement après coup, comme test exploratoire.
- Une bonne accuracy apparente doit être interprétée avec prudence, car les numéros de clusters sont arbitraires.

In [ ]:
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

#Etaape 1 : appliquer KMeans avec k=2 pour simuler le non-supervisé et comparer avec la vraie classe (is_genuine)
from sklearn.cluster import KMeans
import numpy as np

# Utiliser les données ACP (2D) ou les données standardisées (6D)
X = X_scaled  #  X_scaled et non ACP pour tester sur les 6 variables originales affichent des points plus dispersés et moins séparables, mais on peut tester sur les 6 variables originales

# Appliquer K-means avec k=2
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)  # n_init=10 pour éviter les warnings
kmeans.fit(X)
labels_kmeans = kmeans.labels_  # 0 ou 1 (clusters prédits)

# Comparer avec les vraies classes (is_genuine)
print("Répartition des clusters K-means:")
print(np.unique(labels_kmeans, return_counts=True))

⚠️ Attention : K-means ne sait pas que les données ont 2 classes (vrai/faux). Il va juste regrouper les points proches (distance euclidienne). Si les données sont mélangées, K-means peut créer des clusters qui ne correspondent pas à is_genuine !

In [ ]:
plt.scatter(X_scaled[labels_kmeans == 0, 0], X_scaled[labels_kmeans == 0, 1], label='Cluster 0', color='red', alpha=0.7)
plt.scatter(X_scaled[labels_kmeans == 1, 0], X_scaled[labels_kmeans == 1, 1], label='Cluster 1', color='green', alpha=0.7)
plt.xlabel('Composante Principale 1')
plt.ylabel('Composante Principale 2')
plt.title('Clusters K-means (k=2) sur variables standardisées')
plt.legend()
plt.show()

Conclusion : la visualisation montre les groupes construits par K-means, pas directement les classes vrai/faux. Comme K-means raisonne uniquement sur les distances, il peut trouver une structure proche des classes, mais cela ne suffit pas pour en faire un modèle de prédiction fiable en production.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, silhouette_score

# Comparaison exploratoire entre les clusters K-means et les vraies classes
cm = confusion_matrix(y, labels_kmeans)
ari = adjusted_rand_score(y, labels_kmeans)
sil = silhouette_score(X_scaled, labels_kmeans)

print("Confusion Matrix (classes réelles vs clusters K-means):\n", cm)
print(f"Adjusted Rand Index : {ari:.3f}")
print(f"Silhouette score : {sil:.3f}")
print(classification_report(y, labels_kmeans, target_names=['Vrai', 'Faux']))

Lecture : les clusters K-means sont proches des deux classes sur ce jeu de données, mais l'algorithme n'a pas appris la notion de vrai ou faux billet. Cette comparaison reste donc exploratoire. Pour la prédiction finale, il faut privilégier des modèles supervisés entraînés avec la cible `faux_billet`.

#### Conclusion générale
Les algorithmes de clustering ne sont pas retenus pour la prédiction finale : ils sont utiles pour explorer la structure des données, mais ils n'apprennent pas directement la cible métier. Pour détecter les faux billets en production, on poursuit donc avec des modèles supervisés.

### 4.2 Split train / test

#### 4.2.1 Principe et définition du split
Objectif : On entraîne le modèle sur un sous-ensemble des données et on l'évalue sur un jeu jamais vu pour mesurer la généralisation.

In [ ]:
from sklearn.model_selection import train_test_split

# Variables explicatives retenues et cible binaire
FEATURES = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
TARGET = 'faux_billet'

# Filet de sécurité si la cellule de normalisation n'a pas encore été exécutée
if 'df_normalized' not in globals():
    df_normalized = df.copy()

X = df_normalized[FEATURES].copy()
y = df_normalized[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # stratifiy pour conserver la proportion de classes dans les deux ensembles 
 )

In [ ]:
# Vérification des classes dans les ensembles train et test
print(f'Train : {X_train.shape[0]} billets')
print(f'Ratio faux dans train : {(y_train == 1).mean():.1%}')
print(f'Ratio vrai dans train : {(y_train == 0).mean():.1%}')
print(f'Test  : {X_test.shape[0]} billets')
print(f'Ratio faux dans test  : {(y_test == 1).mean():.1%}')
print(f'Ratio vrai dans test  : {(y_test == 0).mean():.1%}')

Attention : le split , lui 80/20 est retenu pour garder suffisamment de données d'entraînement tout en conservant un jeu de test indépendant. La stratification maintient la même proportion de vrais et de faux billets dans les deux ensembles.

#### 4.2.2 Imputation et standardisation
Ces étapes sont apprises sur le train puis appliquées au test pour éviter la fuite de données.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

Précision : scaler.fit_transform(X_train) → scaler.transform(X_test) signifie que l'on apprend les paramètres de normalisation sur le train et on applique la même transformation au test pour éviter la fuite de données. Par contre, on ne peut pas faire scaler.fit_transform(X_test) car cela reviendrait à apprendre les paramètres de normalisation sur le test, ce qui biaiserait notre modèle de classification binaire.

# 5) Modélisation


Pour la classification, il faut utiliser les bonnes métriques pour un problème binaire (faux_billet = 1/0) :
- Accuracy (précision globale).
- Matrice de confusion.
- Precision, Recall, F1-score (surtout si les classes sont déséquilibrées).
- ROC-AUC (pour évaluer la capacité à distinguer les classes).


En résumé : les étapes de prétraitement et de préparation des données sont cruciales pour garantir que le modèle de classification binaire puisse apprendre efficacement à partir des données et généraliser correctement sur de nouvelles observations.

Dans la suite du notebook, on va comparer plusieurs algorithmes supervisés sur les mêmes données préparées afin de retenir le modèle le plus robuste.

#### 5.1 Modèle de classification : régression logistique
La régression logistique sert de baseline supervisée : elle est simple, rapide et interprétable. Elle est adaptée si la séparation entre vrais et faux billets est presque linéaire, mais peut être limitée si les relations entre mesures sont plus complexes.

Ici, la variable is_genuine sert uniquement à l’apprentissage. Une fois le modèle entraîné, il ne voit plus que les caractéristiques du billet et applique ce qu’il a appris pour prédire vrai ou faux.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

# Baseline exportable : régression logistique + prétraitement dans un seul pipeline
FEATURES = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
TARGET = 'faux_billet'

X_model = df[FEATURES].copy()
y_model = df[TARGET].astype(int)

# Split stratifié pour garder la même proportion de classes dans train et test
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.20, random_state=42, stratify=y_model
)

baseline_rl = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=1000)
)

baseline_rl.fit(X_train, y_train)

y_pred = baseline_rl.predict(X_test)
y_pred_proba = baseline_rl.predict_proba(X_test)[:, 1]

print('Accuracy :', accuracy_score(y_test, y_pred))
print('ROC-AUC :', roc_auc_score(y_test, y_pred_proba))
print('Matrice de confusion :\n', confusion_matrix(y_test, y_pred))
print('Rapport de classification :\n', classification_report(y_test, y_pred))

Le jeu de test contient bien 300 billets car j’ai choisi un split 80/20. Les 500 faux billets initiaux sont répartis entre apprentissage et test pour respecter la logique d’évaluation du modèle, logique plus facilement compréhensible et plus robuste que de faire un split 70/30. Cela permet d'avoir un modèle plus robuste et moins sensible aux variations du jeu de test dans les comparaisons entre modèles.

In [ ]:
#affichage de la matrice de confusion et de la courbe ROC
ConfusionMatrixDisplay.from_estimator(baseline_rl, X_test, y_test, display_labels=['Vrai', 'Faux'], cmap=plt.cm.Blues)
plt.title('Matrice de confusion - Régression Logistique')

# Courbe ROC
RocCurveDisplay.from_estimator(baseline_rl, X_test, y_test)

Constat : la régression logistique obtient une accuracy de 0,99, un recall de 0,97 pour la classe `faux_billet` et une ROC-AUC de 0,999. La matrice de confusion indique 3 faux billets prédits comme vrais et aucun vrai billet prédit comme faux sur le jeu de test. C'est une baseline très solide.

Analyse métier : les 3 erreurs importantes sont des faux billets classés comme vrais. Dans un contexte de lutte contre le faux-monnayage, ce type d'erreur est prioritaire à réduire. La suite compare donc plusieurs modèles et privilégie le recall de la classe `faux_billet`.

Conclusion : la régression logistique est performante et interprétable. Elle constitue une bonne référence, mais elle ne doit pas être retenue seule avant comparaison avec d'autres modèles et optimisation par validation croisée.

#### 5.2 Modèle de classification : KNN (K-nearest neighbors)

Le modèle KNN est un modèle simple et efficace pour la classification binaire. Il est particulièrement adapté lorsque les classes sont non linéairement séparables. Dans cette section, il est évalué sur le même split 80/20 que la régression logistique afin de comparer les modèles dans des conditions strictement identiques. Cependant, il peut être limité si les relations entre les variables explicatives et la variable cible sont complexes. Ce qui peut être le cas dans notre jeu de données, où les caractéristiques des billets peuvent interagir de manière complexe pour déterminer leur authenticité.

n_neighbors = 5 est une valeur de référence raisonnable, et elle doit idéalement être confirmée par un petit test de plusieurs valeurs (3, 5, 7, 9) ou une validation croisée.Si n_neighbors est trop petit, le modèle devient trop sensible au bruit; avec un k trop grand, il devient trop lisse et perd en précision locale. Le fait de prendre un nombre impair limite aussi les égalités dans le vote.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Meme split 80/20 que pour la regression logistique: X_train, X_test, y_train, y_test
# Pretraitement integre pour ne pas reutiliser un tableau standardise issu d'un autre split
k = 5
baseline_knn = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=k)
)
baseline_knn.fit(X_train, y_train)

y_pred_knn = baseline_knn.predict(X_test)
y_prob_knn = baseline_knn.predict_proba(X_test)[:, 1]

print(f'KNN avec k={k}')
print('Accuracy KNN :', accuracy_score(y_test, y_pred_knn))
print('Confusion Matrix :\n', confusion_matrix(y_test, y_pred_knn))
print('Classification Report :\n', classification_report(y_test, y_pred_knn))
print('ROC-AUC :', roc_auc_score(y_test, y_prob_knn))


# affichage de la matrice de confusion et de la courbe ROC
ConfusionMatrixDisplay.from_estimator(
    baseline_knn, X_test, y_test,
    display_labels=['Vrai', 'Faux'],
    cmap=plt.cm.Blues # optionnel : couleur de la matrice de confusion
)
plt.title('Matrice de confusion - KNN')
plt.show()

RocCurveDisplay.from_estimator(baseline_knn, X_test, y_test)
plt.title('Courbe ROC - KNN')
plt.show()


Conclusion : le KNN obtient des résultats proches de la régression logistique sur le jeu de test. Comme il dépend fortement de l'échelle des variables et du choix de `n_neighbors`, il sera comparé plus proprement dans la recherche d'hyperparamètres.

#### 5.3 Modèle de classification : Random Forest

Random Forest est testé comme modèle supervisé capable de capter des relations non linéaires entre les mesures du billet. Il est comparé aux autres modèles avec le même split train/test, puis optimisé en section 6.

In [ ]:
# Baseline Random Forest avec le même protocole que les autres modèles
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

baseline_rf = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(class_weight='balanced', random_state=42)
)

baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
y_prob_rf = baseline_rf.predict_proba(X_test)[:, 1]

print('Accuracy RF :', accuracy_score(y_test, y_pred_rf))
print('Confusion Matrix :\n', confusion_matrix(y_test, y_pred_rf))
print('Classification Report :\n', classification_report(y_test, y_pred_rf))
print('ROC-AUC :', roc_auc_score(y_test, y_prob_rf))

ConfusionMatrixDisplay.from_estimator(
    baseline_rf, X_test, y_test,
    display_labels=['Vrai', 'Faux'],
    cmap=plt.cm.Blues
)
plt.title('Matrice de confusion - Random Forest')
plt.show()

RocCurveDisplay.from_estimator(baseline_rf, X_test, y_test)
plt.title('Courbe ROC - Random Forest')
plt.show()

Conclusion : Random Forest obtient également de très bons résultats, avec une ROC-AUC élevée et peu d'erreurs sur le jeu de test. À ce stade, il reste un candidat sérieux, mais le choix final doit être fait après optimisation et validation croisée en section 6.

 ### Remarque sur le Feature Engineering

Un feature engineering avancé pourrait être envisagé, par exemple en créant des écarts entre hauteurs, des ratios de marges ou des indicateurs de symétrie du billet. Cependant, les variables fournies sont déjà des mesures géométriques métier directement pertinentes.

Dans ce projet, le choix a été de privilégier un pipeline simple, robuste et interprétable : imputation, standardisation si nécessaire, comparaison de plusieurs modèles, puis optimisation par validation croisée. Le feature engineering n’est donc pas indispensable ici, d’autant plus que les performances obtenues sont déjà très élevées.

Toute nouvelle variable devrait être testée dans le même protocole de validation croisée afin de vérifier qu’elle améliore réellement la généralisation, sans introduire de surapprentissage.

# 6) Recherche d'hyperparamètres et cross-validation des modèles supervisés

## 6.1 Définition d'une fonction custom_pipeline pour automatiser le processus de modélisation et d'évaluation des modèles de classification binaire.

In [ ]:
# Fonction utilitaire : toutes les étapes du pipeline ont des noms stables.
# Cela permet ensuite d'utiliser les paramètres imputation__..., scaler__..., model__... dans GridSearchCV.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd


def custom_pipeline(model, imputer=None, scaler=None, reducer=None):
    steps = []
    if imputer is not None:
        steps.append(('imputation', imputer))
    if scaler is not None:
        steps.append(('scaler', scaler))
    if reducer is not None:
        steps.append(('reduction', reducer))
    steps.append(('model', model))
    return Pipeline(steps)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

print('Fonction custom_pipeline et validation croisée prêtes')

### Pourquoi refaire une recherche d'hyperparamètres après les premiers modèles ?

L'évaluation modèle par modèle en section 5 sert de **benchmark de base** : on entraîne chaque algorithme avec des paramètres simples pour vérifier rapidement s'il apprend correctement le problème et pour comparer leurs comportements dans les mêmes conditions.

La recherche d'hyperparamètres en section 6 n'est pas strictement obligatoire pour produire une première prédiction, mais elle est fortement recommandée  pour un projet de machine learning plus robuste. Elle permet de vérifier si les performances observées viennent vraiment du modèle, ou seulement d'un choix arbitraire de paramètres comme `n_neighbors` pour KNN, `C` et `penalty` pour la régression logistique, ou `max_depth` et `n_estimators` pour Random Forest.

La fonction `custom_pipeline` définie en 6.1 sert à construire automatiquement des pipelines comparables avec les mêmes étapes :

- `imputation` : remplacement des valeurs manquantes ;
- `scaler` : mise à l'échelle des variables quand le modèle en a besoin ;
- `reduction` : étape optionnelle de réduction de dimension ;
- `model` : algorithme final de classification.

L'intérêt est double : éviter la fuite de données pendant la validation croisée et permettre à `GridSearchCV` de tester proprement plusieurs combinaisons de prétraitement et d'hyperparamètres. La sélection finale repose donc sur une méthode plus défendable qu'un simple choix manuel.


## 6.2 Recherche d'hyper-paramètres et Cross-Validation pour évaluer la robustesse des modèles

#### Interprétation : régularisation et validation croisée

La régularisation limite le surapprentissage : pour la régression logistique, elle passe par les pénalités L1/L2 ; pour Random Forest, elle passe notamment par la profondeur des arbres et les tailles minimales de feuilles ou de divisions.

La validation croisée évalue chaque combinaison d'hyperparamètres sur plusieurs découpages du jeu d'entraînement. `GridSearchCV` permet donc à la fois de tester les paramètres et de sélectionner le meilleur pipeline selon la métrique prioritaire.

#### Dans cette section, on va utiliser la fonction `custom_pipeline` pour tester plusieurs modèles de classification binaire avec différents hyperparamètres. L'objectif est de trouver la combinaison optimale qui maximise les performances sur le jeu de validation tout en évitant le surapprentissage.

In [ ]:
# Définition des pipelines candidats
pipelines = {
    'LogisticRegression': custom_pipeline(
        model=LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        imputer=SimpleImputer(),
        scaler=StandardScaler()
    ),
    'KNN': custom_pipeline(
        model=KNeighborsClassifier(),
        imputer=SimpleImputer(),
        scaler=StandardScaler()
    ),
    'RandomForest': custom_pipeline(
        model=RandomForestClassifier(class_weight='balanced', random_state=42),
        imputer=SimpleImputer(),
        scaler=None
    )
}

Ensuite, nous construisons la grille d’hyperparamètres en respectant les combinaisons autorisées par scikit-learn, pour éviter que GridSearchCV teste des couples impossibles.

In [ ]:
# Grilles d'hyperparamètres compatibles avec les noms d'étapes du pipeline.
# Important : pour LogisticRegression, les couples solver/penalty doivent être compatibles.
param_grids = {
    'LogisticRegression': [
        {
            'imputation__strategy': ['mean', 'median'],
            'scaler': [StandardScaler(), RobustScaler()],
            'model__solver': ['liblinear'],
            'model__penalty': ['l1', 'l2'], 
            'model__C': [0.01, 0.1, 1, 10],
            'model__fit_intercept': [True, False]
        },
        {
            'imputation__strategy': ['mean', 'median'],
            'scaler': [StandardScaler(), RobustScaler()],
            'model__solver': ['lbfgs'],
            'model__penalty': ['l2'],
            'model__C': [0.01, 0.1, 1, 10],
            'model__fit_intercept': [True, False]
        }
    ],
    'KNN': {
        'imputation__strategy': ['mean', 'median'],
        'scaler': [StandardScaler(), RobustScaler()],
        'model__n_neighbors': [3, 5, 7, 9, 11],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['euclidean', 'manhattan']
    },
    'RandomForest': {
        'imputation__strategy': ['mean', 'median'],
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 5, 10, 15],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    }
}

Pour la régression logistique, j’ai séparé les grilles parce que tous les solveurs ne supportent pas toutes les régularisations. Par exemple, liblinear peut tester l1 et l2, alors que lbfgs est utilisé ici uniquement avec l2. Cela évite des combinaisons invalides pendant le GridSearchCV.

In [ ]:
# Exemple : recherche d'hyperparamètres pour la régression logistique seule
lr_grid_search = GridSearchCV(
    estimator=pipelines['LogisticRegression'],
    param_grid=param_grids['LogisticRegression'],
    cv=cv,
    scoring=scoring,
    refit='recall',
    n_jobs=-1,
    return_train_score=True
)

lr_grid_search

In [ ]:
# Exécution de la recherche d'hyperparamètres pour Logistic Regression
lr_grid_search.fit(X_train, y_train)

print(f"Meilleur recall CV : {lr_grid_search.best_score_:.3f}")
print('Meilleurs hyperparamètres :')
print(lr_grid_search.best_params_)

In [ ]:
# Résultats détaillés de la régression logistique
lr_results = pd.DataFrame(lr_grid_search.cv_results_)
lr_results = lr_results.sort_values(by='rank_test_recall')

cols_to_display = [
    'rank_test_recall',
    'mean_test_recall',
    'mean_test_f1',
    'mean_test_roc_auc',
    'mean_test_accuracy',
    'params'
]

display(lr_results[cols_to_display].head(10))

best_lr_model = lr_grid_search.best_estimator_
print('Meilleur pipeline Logistic Regression :')
print(best_lr_model)

#### Recherche commune via GridSearchCV : Logistic Regression, KNN et Random Forest

Rappel : Un fold est un sous-ensemble du jeu d’entraînement utilisé pendant la validation croisée. Avec GridSearchCV, chaque combinaison d'hyperparamètres est évaluée sur plusieurs folds pour obtenir une estimation plus robuste de la performance du modèle. Cela permet de réduire le risque de surapprentissage et d'obtenir un modèle plus généralisable.

Ici, nous avons choisi 5 folds pour la validation croisée, ce qui est un compromis entre robustesse de l'estimation et temps de calcul. Chaque modèle est évalué sur 5 sous-ensembles différents du jeu d'entraînement, et les performances sont moyennées pour obtenir une estimation finale.

Donc avec n_splits=5, chaque observation sert :

plusieurs fois à l’entraînement ;
une fois à la validation.
L’intérêt est d’obtenir une performance plus fiable qu’avec une seule séparation train/test.

In [ ]:
# Recherche d'hyperparamètres pour tous les modèles supervisés
# GridSearchCV fait déjà une cross-validation : chaque combinaison est évaluée sur 5 folds.
grid_searches = {}
summary_rows = []

for model_name, pipeline in pipelines.items():
    print(f'\nRecherche en cours : {model_name}')

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        cv=cv,
        scoring=scoring,
        refit='recall',
        n_jobs=-1,
        return_train_score=True
    )
    grid_search.fit(X_train, y_train)
    grid_searches[model_name] = grid_search

    best_index = grid_search.best_index_
    cv_results = grid_search.cv_results_

    summary_rows.append({
        'modele': model_name,
        'best_recall_cv': cv_results['mean_test_recall'][best_index],
        'best_f1_cv': cv_results['mean_test_f1'][best_index],
        'best_roc_auc_cv': cv_results['mean_test_roc_auc'][best_index],
        'best_accuracy_cv': cv_results['mean_test_accuracy'][best_index],
        'best_params': grid_search.best_params_
    })

cv_summary = pd.DataFrame(summary_rows).sort_values(
    by=['best_recall_cv', 'best_f1_cv', 'best_roc_auc_cv'],
    ascending=False
)

display(cv_summary)

best_model_name = cv_summary.iloc[0]['modele']
best_model = grid_searches[best_model_name].best_estimator_

print(f"Meilleur modèle selon la cross-validation : {best_model_name}")
print('Meilleurs hyperparamètres :')
print(grid_searches[best_model_name].best_params_)

Un fold est une portion du jeu d’entraînement utilisée dans la validation croisée. Avec 5 folds, le modèle est entraîné 5 fois : à chaque fois, 4 folds servent à apprendre et 1 fold sert à valider. Cela permet d’évaluer la stabilité du modèle sur plusieurs découpages des données. La validation croisée est réalisée uniquement sur le jeu d’entraînement. Le jeu de test reste totalement à part pendant le choix des hyperparamètres, puis il est utilisé une seule fois à la fin pour évaluer la capacité du modèle à généraliser sur des données jamais vues.

Conclusion de la cross-validation : le tableau `cv_summary` compare les meilleurs réglages de chaque modèle avec le même protocole de validation croisée. Le critère principal retenu est le `recall`, car dans un problème de détection de faux billets, on veut surtout éviter de laisser passer des faux billets. En cas d'égalité, on départage avec le F1-score puis la ROC-AUC. Le pipeline retenu est stocké dans `best_model` et pourra être évalué sur le jeu de test dans la section suivante.

# 7) Sélection du meilleur modèle pour la détection de faux billets

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Evaluation finale sur le jeu de test, jamais utilisé pour choisir les hyperparamètres
if 'best_model' not in globals():
    raise ValueError("Exécute d'abord la section 6 pour créer best_model avec GridSearchCV.")

y_pred_test = best_model.predict(X_test)
y_prob_test = best_model.predict_proba(X_test)[:, 1]

test_scores = pd.DataFrame([{
    'modele': best_model_name,
    'accuracy_test': accuracy_score(y_test, y_pred_test),
    'precision_test': precision_score(y_test, y_pred_test, zero_division=0), # zero_division=0 pour éviter les erreurs si aucune prédiction positive n'est faite
    'recall_test': recall_score(y_test, y_pred_test, zero_division=0),
    'f1_test': f1_score(y_test, y_pred_test, zero_division=0),
    'roc_auc_test': roc_auc_score(y_test, y_prob_test)
}])

display(test_scores)

print('Matrice de confusion sur le test :')
print(confusion_matrix(y_test, y_pred_test))
print('\nRapport de classification sur le test :')
print(classification_report(y_test, y_pred_test, target_names=['Vrai', 'Faux']))

## 7.1 Vérification du surapprentissage

Objectif : comparer les performances train/test du best_model.

In [ ]:
#Vérification du surapprentissage : comparaison des scores CV et test
cv_summary['accuracy_test'] = test_scores['accuracy_test'].values[0]
cv_summary['precision_test'] = test_scores['precision_test'].values[0]
cv_summary['recall_test'] = test_scores['recall_test'].values[0]
cv_summary['f1_test'] = test_scores['f1_test'].values[0]

display(cv_summary)

## 7.2 Distribution des probabilités prédites

Objectif : regarder si les vrais billets ont une probabilité de faux proche de 0, et les faux billets une probabilité de faux proche de 1.

In [ ]:
# Distribution des probabilités prédites pour la classe positive (faux billet) sur le jeu de test
#y_prob_test = probabilités prédites par le meilleur modèle sur X_test
#y_test = vraies classes du jeu de test

plt.figure(figsize=(8, 6))

sns.histplot(
    y_prob_test[y_test == 0], # on sélectionne les probabilités prédites pour les vrais billets.
    color='blue',
    label='Vrai billet',
    kde=True,
    stat='density'
)

sns.histplot(
    y_prob_test[y_test == 1], # on sélectionne les probabilités prédites pour les faux billets.
    color='red',
    label='Faux billet',
    kde=True,
    stat='density'
)

plt.xlabel('Probabilité prédite de faux billet')
plt.ylabel('Densité')
plt.title('Distribution des probabilités prédites par classe')
plt.legend()
plt.show()

Conclusion : la distribution des probabilités prédites montre que le modèle est capable de distinguer les vrais et les faux billets. Les vrais billets ont une probabilité de faux proche de 0, tandis que les faux billets ont une probabilité de faux proche de 1. Cela confirme que le modèle a appris à identifier correctement les caractéristiques des billets authentiques et contrefaits.

Après avoir sélectionné le meilleur pipeline, je vérifie qu’il ne surapprend pas et j’analyse ses probabilités prédites. Cela permet de compléter les métriques classiques par une lecture plus qualitative du comportement du modèle.

# 8) Construction du pipeline final 

In [ ]:
# Pipeline final retenu après recherche d'hyperparamètres et validation croisée
pipe = best_model

print(f'Pipeline final retenu : {best_model_name}')
pipe

# 9) Sauvegarde du meilleur modèle

Le modèle déployé doit inclure le prétraitement (imputation et standardisation) pour garantir des prédictions cohérentes en production.

In [ ]:
from sklearn.base import clone
import joblib
from pathlib import Path

# Réentraîner le meilleur pipeline sur toutes les données disponibles avant déploiement
if 'best_model' not in globals():
    raise ValueError("Exécute d'abord les sections 6 et 7 pour sélectionner best_model.")

X_final = df[FEATURES].copy()
y_final = df[TARGET].astype(int)

final_model = clone(best_model)
final_model.fit(X_final, y_final)

output_path = Path('../src/model_oncfm.joblib')
joblib.dump(final_model, output_path)

print(f'Modèle sauvegardé : {output_path.resolve()}')
print(f'Modèle retenu : {best_model_name}')
print('Pipeline exporté :')
print(final_model)